In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, accuracy_score, log_loss
import warnings
from tqdm import tqdm
import os
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/IrrigationNeed/')

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
irrigation_data = pd.read_csv("train.csv", index_col = 0)
irrigation_data

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High


In [3]:
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
transformer = ColumnTransformer(transformers=[('ohe', ohe, make_column_selector(dtype_include=object))], remainder='passthrough', verbose_feature_names_out=False).set_output(transform = 'pandas')

In [4]:
irrigation_data.isna().sum().sum()

0

In [5]:
X,y = irrigation_data.drop('Irrigation_Need',axis=1), irrigation_data['Irrigation_Need']

In [6]:
le = LabelEncoder()
ss = StandardScaler()
xgb = XGBClassifier(random_state=26)
pca = PCA()
y = le.fit_transform(y)

In [7]:
pipe = Pipeline([('OHE',transformer),('SS',ss),('PCA',pca),('XGB',xgb)])
# pipe.get_params()

In [8]:
params = {"XGB__max_depth" : [2, 3, 4, 5], "XGB__learning_rate" : np.linspace(0.05, 0.7, 5), "XGB__n_estimators" : [10, 25,50,75]}
folds = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 26)

gcv = GridSearchCV(estimator = pipe, cv = folds, param_grid = params, n_jobs = -1, verbose = 3, scoring='balanced_accuracy')
gcv.fit(X, y)


Fitting 5 folds for each of 80 candidates, totalling 400 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=26, shuffle=True),
             estimator=Pipeline(steps=[('OHE',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7ffa9442aac0>)],
                                                          verbose_feature_names_out=...
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None,
                                                      random_state=26, ...))]),
             n_jobs=-1,
             param_grid={'XGB__learning_rate': array([0.05  , 0.2125, 0.375 , 0.5375, 0.7   ]),
                         'XGB__max_depth': [2, 3, 4, 5],
                         'XGB__n_estimators': [10, 25, 50, 75]},
             scoring='balanced_accuracy', verbose=3)

In [11]:
gcv.best_estimator_, gcv.best_score_

(Pipeline(steps=[('OHE',
                  ColumnTransformer(remainder='passthrough',
                                    transformers=[('ohe',
                                                   OneHotEncoder(drop='first',
                                                                 handle_unknown='ignore',
                                                                 sparse_output=False),
                                                   <sklearn.compose._column_transformer.make_column_selector object at 0x7ffafb919b50>)],
                                    verbose_feature_names_out=False)),
                 ('SS', StandardScaler()), ('PCA', PCA()),
                 ('XGB',
                  XGBClassifier(base_score=None,...
                                feature_types=None, gamma=None, grow_policy=None,
                                importance_type=None,
                                interaction_constraints=None, learning_rate=0.7,
                                max_bi

[CV 1/5] END XGB__learning_rate=0.05, XGB__max_depth=2, XGB__n_estimators=75;, score=0.556 total time=  38.7s
[CV 2/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=75;, score=0.594 total time=  43.7s
[CV 4/5] END XGB__learning_rate=0.05, XGB__max_depth=4, XGB__n_estimators=75;, score=0.635 total time=  47.1s
[CV 5/5] END XGB__learning_rate=0.05, XGB__max_depth=5, XGB__n_estimators=75;, score=0.667 total time=  54.7s
[CV 1/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=3, XGB__n_estimators=50;, score=0.673 total time=  32.9s
[CV 2/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=4, XGB__n_estimators=25;, score=0.653 total time=  24.4s
[CV 1/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=5, XGB__n_estimators=10;, score=0.641 total time=  19.5s
[CV 2/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=5, XGB__n_estimators=25;, score=0.686 total time=  26.5s
[CV 2/5] END XGB__learning_rate=0.37499999999999994, XGB__ma

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


[CV 4/5] END XGB__learning_rate=0.05, XGB__max_depth=2, XGB__n_estimators=25;, score=0.519 total time=  22.8s
[CV 4/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=10;, score=0.529 total time=  18.3s
[CV 5/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=50;, score=0.584 total time=  32.4s
[CV 4/5] END XGB__learning_rate=0.05, XGB__max_depth=4, XGB__n_estimators=50;, score=0.618 total time=  35.8s
[CV 5/5] END XGB__learning_rate=0.05, XGB__max_depth=5, XGB__n_estimators=25;, score=0.633 total time=  26.7s
[CV 4/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=10;, score=0.538 total time=  16.6s
[CV 3/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=50;, score=0.628 total time=  29.7s
[CV 5/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=3, XGB__n_estimators=10;, score=0.574 total time=  18.5s
[CV 3/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=3, XGB_

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


...

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain featu

[CV 3/5] END XGB__learning_rate=0.05, XGB__max_depth=2, XGB__n_estimators=50;, score=0.530 total time=  30.8s
[CV 1/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=50;, score=0.582 total time=  33.2s
[CV 3/5] END XGB__learning_rate=0.05, XGB__max_depth=4, XGB__n_estimators=25;, score=0.589 total time=  24.2s
[CV 5/5] END XGB__learning_rate=0.05, XGB__max_depth=4, XGB__n_estimators=75;, score=0.635 total time=  49.1s
[CV 1/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=25;, score=0.578 total time=  22.0s
[CV 2/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=75;, score=0.656 total time=  37.7s
[CV 2/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=3, XGB__n_estimators=75;, score=0.704 total time=  41.6s
[CV 4/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=4, XGB__n_estimators=75;, score=0.751 total time=  47.2s
[CV 1/5] END XGB__learning_rate=0.37499999999999994, XGB__ma

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


[CV 1/5] END XGB__learning_rate=0.05, XGB__max_depth=2, XGB__n_estimators=25;, score=0.514 total time=  22.7s
[CV 3/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=10;, score=0.523 total time=  16.5s
[CV 4/5] END XGB__learning_rate=0.05, XGB__max_depth=3, XGB__n_estimators=50;, score=0.585 total time=  32.3s
[CV 1/5] END XGB__learning_rate=0.05, XGB__max_depth=4, XGB__n_estimators=50;, score=0.616 total time=  37.3s
[CV 3/5] END XGB__learning_rate=0.05, XGB__max_depth=5, XGB__n_estimators=25;, score=0.623 total time=  27.0s
[CV 3/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=10;, score=0.529 total time=  18.3s
[CV 2/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=2, XGB__n_estimators=50;, score=0.623 total time=  31.3s
[CV 4/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=3, XGB__n_estimators=25;, score=0.622 total time=  24.7s
[CV 4/5] END XGB__learning_rate=0.21249999999999997, XGB__max_depth=4, XGB_

/home/pgcp-ai/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [9]:
tst = pd.read_csv("test.csv", index_col = 0)
tst

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,Yes,43.49,East
899996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,Yes,92.03,West
899997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,Yes,34.02,East


In [10]:
tst_ohe = transformer.transform(tst)

NotFittedError: This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
tst_ss = ss.transform(tst_ohe)
tst_ss

In [ ]:
tst["Irrigation_Need"] = le.inverse_transform(bm.predict(tst_ss))
tst

In [ ]:
submit = pd.read_csv("sample_submission.csv")
submit

In [ ]:
submit["Irrigation_Need"] = le.inverse_transform(bm.predict(tst_ss))
submit

In [ ]:
submit.to_csv("KaggleSubmissionXGB.csv", index = False)